# 01 — Exploratory Data Analysis
**Healthcare Operations Analytics — Raw Data Profiling**

Before any cleaning happens, this notebook profiles the raw extract exactly as it would come out of the hospital's admissions system: nulls, duplicates, and a few clock-sync artifacts included. The goal here is to *find* the problems, not fix them — cleaning happens in `02_data_cleaning.ipynb`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)

admissions = pd.read_csv('../data/raw/admissions.csv', parse_dates=['admission_date', 'discharge_date'])
patients = pd.read_csv('../data/raw/patients.csv')
departments = pd.read_csv('../data/raw/departments.csv')
billing = pd.read_csv('../data/raw/billing.csv')

print(f"Admissions (raw): {len(admissions):,} rows")
print(f"Patients: {len(patients):,} rows")
admissions.head(3)

Admissions (raw): 47,058 rows
Patients: 18,000 rows


,admission_id,patient_id,doctor_id,department_id,admission_date,admission_type,severity_score,length_of_stay_days,discharge_date,wait_minutes,discharge_efficiency,readmitted_30d,department_raw_text
0,23509,6672,136,5,2023-03-10,Emergency,3.8,3.4,2023-03-13 09:36:00,106.0,65.7,0,Pediatrics
1,26138,16754,4,9,2023-09-05,Urgent,3.0,7.1,2023-09-12 02:24:00,15.0,66.3,1,Obstetrics
2,18329,6959,65,4,2025-05-08,Emergency,6.3,7.9,2025-05-15 21:36:00,28.0,100.0,0,Oncology


## Shape & dtypes check

In [2]:
admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 47058 entries, 0 to 47057
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   admission_id          47058 non-null  int64         
 1   patient_id            47058 non-null  int64         
 2   doctor_id             47058 non-null  int64         
 3   department_id         47058 non-null  int64         
 4   admission_date        47058 non-null  datetime64[us]
 5   admission_type        47058 non-null  str           
 6   severity_score        47058 non-null  float64       
 7   length_of_stay_days   47058 non-null  float64       
 8   discharge_date        47058 non-null  datetime64[us]
 9   wait_minutes          47058 non-null  float64       
 10  discharge_efficiency  47058 non-null  float64       
 11  readmitted_30d        47058 non-null  int64         
 12  department_raw_text   47058 non-null  str           
dtypes: datetime64[us](2), float

## Missingness

In [3]:
missing = admissions.isna().sum()
missing = missing[missing > 0]
print(missing)
print()
print("Patient intake gaps:")
print(patients.isna().sum())

Series([], dtype: int64)

Patient intake gaps:
patient_id          0
patient_name        0
date_of_birth       0
gender              0
state             360
insurance_type    720
dtype: int64


## Duplicate rows

The admissions system is known to double-fire on a small fraction of records when
a nurse re-submits after a timeout. We check for exact duplicates on the natural key
(patient, doctor, department, admission date, admission type) rather than the surrogate
`admission_id`, since a re-submitted record gets a *new* ID but is otherwise identical.

In [4]:
dupe_mask = admissions.duplicated(
    subset=['patient_id','doctor_id','department_id','admission_date','admission_type'], keep=False
)
print(f"Rows involved in duplicate submissions: {dupe_mask.sum()}")
admissions[dupe_mask].sort_values(['patient_id','admission_date']).head(6)

Rows involved in duplicate submissions: 1116


,admission_id,patient_id,doctor_id,department_id,admission_date,admission_type,severity_score,length_of_stay_days,discharge_date,wait_minutes,discharge_efficiency,readmitted_30d,department_raw_text
21834,28800,39,32,5,2023-09-11,Urgent,7.0,3.5,2023-09-14 12:00:00,14.0,75.5,0,Pediatrics
24676,28800,39,32,5,2023-09-11,Urgent,7.0,3.5,2023-09-14 12:00:00,14.0,75.5,0,Pediatrics
4151,14915,47,130,2,2025-02-24,Emergency,5.3,3.2,2025-02-27 04:48:00,39.0,72.9,0,Emergency Dept
21313,14915,47,130,2,2025-02-24,Emergency,5.3,3.2,2025-02-27 04:48:00,39.0,72.9,0,Emergency Dept
18791,43550,70,133,2,2025-04-02,Urgent,5.9,4.6,2025-04-06 14:24:00,13.0,93.1,0,Emergency
33872,43550,70,133,2,2025-04-02,Urgent,5.9,4.6,2025-04-06 14:24:00,13.0,93.1,0,Emergency


## Department name inconsistency (legacy system merge)

In [5]:
print(admissions['department_raw_text'].value_counts())

department_raw_text
Pediatrics         5552
General Surgery    5180
Orthopedics        4810
Primary Care       4127
Oncology           3798
Obstetrics         3609
Neurology          3221
Emergency          2778
Emergency Dept     2771
ER                 2685
ICU                2665
cardiology         1990
CARDIOLOGY         1968
Cardiology         1904
Name: count, dtype: int64


## Wait time distribution — including the clock-sync bug tail

In [6]:
print(admissions['wait_minutes'].describe())
print()
neg = (admissions.wait_minutes <= 0).sum()
print(f"Non-positive wait_minutes values (data bug, not real triage times): {neg} "
      f"({neg/len(admissions)*100:.2f}% of rows)")

count    47058.000000
mean        24.029389
std         16.433039
min        -12.000000
25%         12.000000
50%         21.000000
75%         32.000000
max        177.000000
Name: wait_minutes, dtype: float64

Non-positive wait_minutes values (data bug, not real triage times): 281 (0.60% of rows)


## Numeric field summary

In [7]:
admissions[['severity_score','length_of_stay_days','wait_minutes','discharge_efficiency']].describe().round(2)

,severity_score,length_of_stay_days,wait_minutes,discharge_efficiency
count,47058.00,47058.00,47058.00,47058.00
mean,5.21,5.26,24.03,68.57
std,1.78,2.62,16.43,15.36
min,1.00,0.50,-12.00,10.00
25%,4.00,3.30,12.00,58.20
50%,5.20,4.70,21.00,68.80
75%,6.40,6.70,32.00,79.40
max,10.00,23.10,177.00,100.00


## Admission volume by department (raw text, pre-normalization)

In [8]:
dept_counts = admissions['department_raw_text'].value_counts()
dept_counts

department_raw_text
Pediatrics         5552
General Surgery    5180
Orthopedics        4810
Primary Care       4127
Oncology           3798
Obstetrics         3609
Neurology          3221
Emergency          2778
Emergency Dept     2771
ER                 2685
ICU                2665
cardiology         1990
CARDIOLOGY         1968
Cardiology         1904
Name: count, dtype: int64

## Takeaways for the cleaning notebook

1. **558 duplicate admission rows** from double-submission — need dedup on natural key, not `admission_id`.
2. **~0.6% of `wait_minutes` values are non-positive** — a triage clock-sync bug, not real data. Needs correction, not deletion (removing them would bias average wait time downward by discarding otherwise-valid admissions).
3. **Department names have 3 casing/naming variants** for Cardiology and Emergency from a legacy system merge — must be joined back to the canonical `department_id`, which (fortunately) was preserved correctly upstream even where the raw text wasn't.
4. **Insurance type (~4%) and state (~2%) missing** on patient intake — acceptable to leave as `NULL`/"Unknown" category rather than impute, since imputing insurance type would fabricate financial risk data.
